# CLIP–Jev：现有 2×2 模型的两套评价集统一评估

**运行目标**：读取已有 raw / bias-only / temperature-only / combined 四条件、每条件三种子的 checkpoint，在历史 1000 条和补充 200 条上统一评价。**不训练、不重新选 epoch、不拟合温度、不调用 Teacher/CLIP**。训练温度 τ=2 不用于评价输出。

推荐 Colab CPU 运行时；只运行小型 MLP，无需 GPU。也支持 GPU，但同一次运行统一设备，且必须通过开发集重放。

准备两个输入（ZIP 或已解压的根目录均可）：
1. 你已完成的 `CLIP_Jev_Ablation_2x2_20261028...zip`，必须包含12个 best.pt。
2. `Stage2_统一评价_协议与输入_20261026.zip`，含两套冻结特征、开发重放特征和固定抽样数组。也支持旧统一评价结果包中含完整输入的 inputs 目录。

将它们放在 Google Drive，修改下面两个输入路径，然后从上到下运行。只上传这个 notebook 即可；评价代码与已核对输入哈希均已内嵌。

**解释边界**：两套评价集已被查看，本次属于后续探索。所有成分比较及交互区间均未校正多重比较，条件于固定三个训练种子。区间跨0不证明等效，交互不跨0也不证明内部机制。此轮先完成原始输出的四条件比较；不混入旧模型的Student TS结果。

## 1. 挂载 Google Drive（本地运行可跳过）

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('本地环境：无需挂载 Google Drive。')

## 2. 修改路径
右键 Drive 中的文件复制路径；也可以填直接包含 protocol.json / evaluation_protocol_LOCKED.json 的解压目录。OUTPUT_DIR 每次自动生成新目录，结果保留在 Drive。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

ABLATION_INPUT = '/content/drive/MyDrive/CLIP_Jev_Ablation_2x2_20261028-20261028T084446Z-1-001.zip'
EVALUATION_INPUT = '/content/drive/MyDrive/Stage2_统一评价_协议与输入_20261026.zip'
OUTPUT_DIR = '/content/drive/MyDrive/CLIP_Jev_2x2_Evaluation_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DEVICE = 'cpu'  # 可改为 'cuda'；CPU 足够，只运行已冻结特征上的 MLP

for label, path in [('消融模型', ABLATION_INPUT), ('评价输入', EVALUATION_INPUT)]:
    if not Path(path).exists():
        raise FileNotFoundError(f'{label}路径不存在：{path}。请修改本单元格中的路径。')
print('输出目录：', OUTPUT_DIR)

## 3. 检查依赖
Colab 通常已包含所需库。若提示缺失，只安装提示中的包，再重新运行本单元格。不要为了使结果通过而放宽重放阈值。

In [ ]:
import importlib.util
needed = ['numpy', 'torch', 'matplotlib']
missing = [n for n in needed if importlib.util.find_spec(n) is None]
if missing:
    raise ImportError('缺少依赖：' + ', '.join(missing) + '。在新单元格运行 %pip install ' + ' '.join(missing))
import torch, numpy as np
print('PyTorch:', torch.__version__, 'NumPy:', np.__version__)
print('设备：', DEVICE, '；本次不作速度比较。')

## 4. 加载本 notebook 的评价实现与固定输入指纹
这格较长，不需要修改。不会导入模型压缩包中的 Python 代码；checkpoint 使用 weights_only=True 加载。

In [ ]:
MODULE_SOURCE = '"""Inference-only 2x2 evaluation. No source-package code is imported."""\nfrom pathlib import Path, PurePosixPath\nimport csv, io, json, hashlib, platform, zipfile, os, datetime\nimport numpy as np\n\nMETHODS=[\'KD_raw\',\'KD_bias\',\'KD_temperature\',\'KD_calibrated\']\nSEEDS=[20261010,20261011,20261012]\nMETRICS=[\'macro_f1\',\'accuracy\',\'nll\',\'ece10\',\'brier\',\'f1_E\',\'f1_N\',\'f1_C\']\nCONTRASTS={\n \'bias_at_T1\':[ -1,1,0,0], \'bias_at_Tfull\':[0,0,-1,1],\n \'temperature_at_b0\':[-1,0,1,0], \'temperature_at_bfull\':[0,-1,0,1],\n \'full_minus_raw\':[-1,0,0,1], \'interaction\':[1,-1,-1,1]}\ndef require(ok,msg):\n    if not ok:raise ValueError(msg)\ndef sha(b):return hashlib.sha256(b).hexdigest()\ndef digest_file(p):\n    h=hashlib.sha256()\n    with Path(p).open(\'rb\') as f:\n        for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\ndef save(p,obj):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    p.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding=\'utf-8\')\ndef table(p,records):\n    p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n    require(bool(records),\'Empty output: \'+str(p))\n    with p.open(\'w\',encoding=\'utf-8-sig\',newline=\'\') as f:\n        w=csv.DictWriter(f,fieldnames=list(records[0]));w.writeheader();w.writerows(records)\ndef csvbytes(b):return list(csv.DictReader(io.StringIO(b.decode(\'utf-8-sig\'))))\n\nclass Package:\n    """Read selected members without extracting or executing supplied programs."""\n    def __init__(self,path,marker):\n        self.path=Path(path).expanduser().resolve();require(self.path.exists(),\'输入不存在：\'+str(self.path))\n        self.z=zipfile.ZipFile(self.path) if self.path.is_file() else None\n        if self.z:\n            names=self.z.namelist();require(len(names)==len(set(names)),\'ZIP成员重复\')\n            candidates=[n[:-len(marker)] for n in names if n==marker or n.endswith(\'/\'+marker)]\n        else:\n            candidates=[str(q.parents[len(PurePosixPath(marker).parts)-1].relative_to(self.path)).replace(\'\\\\\',\'/\')+\'/\' for q in self.path.rglob(marker)]\n            candidates=[\'\' if x==\'./\' else x for x in candidates]\n        require(len(candidates)==1,\'未唯一定位 \'+marker+\'；请指定该包或其直接根目录。\')\n        self.prefix=candidates[0]\n    def read(self,name):\n        require(not PurePosixPath(name).is_absolute() and \'..\' not in PurePosixPath(name).parts and \'\\\\\' not in name,\'非法包内路径\')\n        if self.z:return self.z.read(self.prefix+name)\n        q=(self.path/self.prefix/name).resolve();require(q.is_relative_to(self.path),\'路径越界\');return q.read_bytes()\n    def j(self,name):return json.loads(self.read(name))\n    def npz(self,name):\n        with np.load(io.BytesIO(self.read(name)),allow_pickle=False) as a:return {k:a[k].copy() for k in a.files}\n\ndef metrics(z,y):\n    z=np.asarray(z,dtype=np.float64);y=np.asarray(y,dtype=np.int64)\n    require(z.shape==(len(y),3) and np.isfinite(z).all() and np.isin(y,[0,1,2]).all(),\'非法预测/标签\')\n    m=z.max(1);lse=m+np.log(np.exp(z-m[:,None]).sum(1));p=np.exp(z-lse[:,None]);pred=z.argmax(1)\n    cm=np.bincount(3*y+pred,minlength=9).reshape(3,3);f=2*cm.diagonal()/np.maximum(cm.sum(0)+cm.sum(1),1)\n    conf=p.max(1);hit=pred==y;bins=np.minimum((conf*10).astype(int),9)\n    nll=lse-z[np.arange(len(y)),y];brier=((p-np.eye(3)[y])**2).sum(1)\n    vals=dict(macro_f1=float(f.mean()),accuracy=float(hit.mean()),nll=float(nll.mean()),ece10=float(sum(abs(((hit-conf)*(bins==b)).mean()) for b in range(10))),brier=float(brier.mean()),f1_E=float(f[0]),f1_N=float(f[1]),f1_C=float(f[2]))\n    return vals,p,pred,nll,brier\n\ndef bootstrap_scores(z,y,draws,chunk=250):\n    vals,p,pred,nll,brier=metrics(z,y);n=len(y);conf=p.max(1);bins=np.minimum((conf*10).astype(int),9)\n    A=np.column_stack([np.eye(9)[y*3+pred],nll,brier]+[(pred==y).astype(float)*(bins==b)-conf*(bins==b) for b in range(10)])\n    blocks=[]\n    for start in range(0,len(draws),chunk):\n        ix=draws[start:start+chunk];offset=np.arange(len(ix))[:,None]*n\n        W=np.bincount((ix+offset).ravel(),minlength=len(ix)*n).reshape(len(ix),n)\n        blocks.append(W@A)\n    a=np.concatenate(blocks);cm=a[:,:9].reshape(-1,3,3)\n    f=2*np.diagonal(cm,axis1=1,axis2=2)/np.maximum(cm.sum(1)+cm.sum(2),1)\n    scores=dict(macro_f1=f.mean(1),accuracy=np.trace(cm,axis1=1,axis2=2)/n,nll=a[:,9]/n,brier=a[:,10]/n,ece10=abs(a[:,11:]).sum(1)/n,f1_E=f[:,0],f1_N=f[:,1],f1_C=f[:,2])\n    for ix in sorted(set([0,min(123,len(draws)-1),len(draws)-1])):\n        direct=metrics(z[draws[ix]],y[draws[ix]])[0]\n        require(all(abs(direct[k]-scores[k][ix])<1e-10 for k in METRICS),\'Bootstrap direct validation failed\')\n    return scores\n\ndef prepare(ablation_path,eval_path,out,anchors,device=\'cpu\'):\n    import torch\n    ab=Package(ablation_path,\'report/status.json\')\n    # marker has a subdirectory: prefix must include only the package root\n    ev=Package(eval_path,\'evaluation_protocol_LOCKED.json\')\n    out=Path(out).resolve()\n    require(not (out/\'completion.json\').exists(),\'此目录已完成。请保留已有结果，并为新运行设置新的 OUTPUT_DIR。\')\n    for source in [ab.path,ev.path]:require(out!=source and (not source.is_dir() or not out.is_relative_to(source)),\'输出目录须独立于输入\')\n    require(sha(ab.read(\'protocol.json\'))==anchors[\'ablation_protocol_sha256\'],\'2×2协议不是已核对的版本\')\n    protocol=ab.j(\'protocol.json\');require(ab.j(\'report/status.json\')[\'complete\'],\'2×2未全部完成\')\n    require(sha(ev.read(\'evaluation_protocol_LOCKED.json\'))==anchors[\'evaluation_protocol_sha256\'],\'评价协议版本不符\')\n    manifest_hashes=ev.j(\'PACKAGE_SHA256.json\')\n    require(sha(ev.read(\'PACKAGE_SHA256.json\'))==anchors[\'evaluation_manifest_sha256\'],\'评价输入清单版本不符\')\n    def verified(name):\n        b=ev.read(name);require(name in manifest_hashes and sha(b)==manifest_hashes[name],\'评价输入哈希不符：\'+name);return b\n    datasets={}\n    for spec in ev.j(\'evaluation_protocol_LOCKED.json\')[\'datasets\']:\n        split=spec[\'split\'];b=verified(spec[\'features_file\']);require(sha(b)==spec[\'npz_sha256\'],\'特征哈希不符\')\n        with np.load(io.BytesIO(b),allow_pickle=False) as a:d={k:a[k].copy() for k in [\'x\',\'y\',\'ids\',\'image_ids\']}\n        rows=[json.loads(s) for s in verified(spec[\'manifest_file\']).decode(\'utf-8-sig\').splitlines() if s.strip()]\n        require(sha(ev.read(spec[\'manifest_file\']))==spec[\'manifest_sha256\'],\'子清单哈希不符\')\n        require(len(rows)==len(d[\'y\'])==spec[\'rows\'] and len(set(map(str,d[\'image_ids\'])))==len(rows),\'样本数/图片唯一性不符\')\n        require(d[\'ids\'].tolist()==[r[\'sample_id\'] for r in rows] and d[\'y\'].tolist()==[r[\'label\'] for r in rows] and list(map(str,d[\'image_ids\']))==[str(r[\'image_id\']) for r in rows],\'ID/标签/图片顺序不符\')\n        x=d[\'x\'];require(x.dtype==np.float32 and x.shape==(len(rows),2048) and np.isfinite(x).all(),\'特征维度/精度错误\')\n        v,t=x[:,:512],x[:,512:1024]\n        require(np.allclose(np.linalg.norm(v,axis=1),1,atol=1e-4) and np.allclose(np.linalg.norm(t,axis=1),1,atol=1e-4),\'特征未归一化\')\n        require(np.allclose(x[:,1024:1536],abs(v-t),atol=1e-6) and np.allclose(x[:,1536:],v*t,atol=1e-6),\'特征组合错误\')\n        with np.load(io.BytesIO(verified(f\'bootstrap/{split}_draws.npz\')),allow_pickle=False) as a:\n            draws=a[\'indices\'].copy();ordered=list(map(str,a[\'image_ids\']))\n        require(draws.shape==(10000,len(rows)) and np.issubdtype(draws.dtype,np.integer) and draws.min()>=0 and draws.max()<len(rows),\'抽样数组错误\')\n        require(len(set(ordered))==len(rows) and set(ordered)==set(map(str,d[\'image_ids\'])),\'抽样图片ID错误\')\n        lookup={str(s):i for i,s in enumerate(d[\'image_ids\'])};d[\'order\']=np.array([lookup[s] for s in ordered]);d[\'draws\']=draws\n        datasets[split]=d\n    require(set(datasets)=={\'historical_test\',\'new_dev_holdout\'},\'评价集不完整\')\n    require(not set(map(str,datasets[\'historical_test\'][\'image_ids\']))&set(map(str,datasets[\'new_dev_holdout\'][\'image_ids\'])),\'评价集图片重叠\')\n    with np.load(io.BytesIO(verified(\'features/val_select_replay.npz\')),allow_pickle=False) as a:dev={k:a[k].copy() for k in a.files}\n    devrows=[json.loads(s) for s in verified(\'manifests/val_select.jsonl\').decode(\'utf-8-sig\').splitlines() if s.strip()]\n    require(dev[\'x\'].shape==(400,2048),\'重放特征错误\')\n    if \'ids\' in dev:require(dev[\'ids\'].tolist()==[r[\'sample_id\'] for r in devrows],\'重放顺序错误\')\n    models=[]\n    for method in METHODS:\n        group=f\'groups/{method}_alpha0.75\';gp=ab.j(group+\'/protocol.json\');gsha=sha(ab.read(group+\'/protocol.json\'))\n        require(gp[\'parent_sha256\']==anchors[\'ablation_protocol_sha256\'],\'模型父协议错误\')\n        expected=dict(b=protocol[\'teacher_params\'][\'b\'] if method in [\'KD_bias\',\'KD_calibrated\'] else 0.,T=protocol[\'teacher_params\'][\'T\'] if method in [\'KD_temperature\',\'KD_calibrated\'] else 1.)\n        require(gp[\'teacher_params\']==expected and gp[\'config\'][\'lr\']==.0003 and gp[\'config\'][\'alpha\']==.75 and gp[\'config\'][\'tau\']==2.,\'2×2配置不符\')\n        for seed in SEEDS:\n            mid=f\'{method}_{seed}\';folder=group+\'/runs/\'+mid;done=ab.j(folder+\'/done.json\');ref=anchors[\'models\'][mid]\n            require(done[\'identity\']==dict(protocol_sha256=gsha,method=method,seed=seed),\'模型身份错配\')\n            b=ab.read(folder+\'/best.pt\');require(sha(b)==done[\'checkpoint_sha256\']==ref[\'checkpoint_sha256\'],\'checkpoint哈希不符：\'+mid)\n            history=ab.read(folder+\'/history.json\');require(sha(history)==done[\'history_sha256\'],\'训练历史哈希不符\')\n            hist=json.loads(history);require([r[\'epoch\'] for r in hist]==list(range(1,31)),\'训练轮数不全\')\n            best=max(hist,key=lambda r:(r[\'validation\'][\'macro_f1\'],-r[\'validation\'][\'nll\'],-r[\'epoch\']))\n            require(best[\'epoch\']==done[\'best_epoch\']==ref[\'best_epoch\'],\'选择规则不符\')\n            preds=ab.read(folder+\'/predictions.csv\');require(sha(preds)==ref[\'predictions_sha256\'],\'开发预测版本错误\')\n            rr=csvbytes(preds);require([r[\'sample_id\'] for r in rr]==[r[\'sample_id\'] for r in devrows] and [int(r[\'label\']) for r in rr]==[r[\'label\'] for r in devrows],\'开发预测ID/标签不符\')\n            models.append(dict(method=method,seed=seed,model_id=mid,checkpoint_sha256=sha(b),best_epoch=done[\'best_epoch\'],payload=b,reference=np.array([[float(r[\'logit_\'+k]) for k in \'ENC\'] for r in rr])))\n    require(device in [\'cpu\',\'cuda\'] and (device==\'cpu\' or torch.cuda.is_available()),\'所选设备不可用\')\n    torch.set_num_threads(2);torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True\n    torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False;torch.use_deterministic_algorithms(True)\n    env=dict(torch=str(torch.__version__),numpy=np.__version__,python=platform.python_version(),device=device,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0) if device==\'cuda\' else None,dtype=\'float32\',batch_size=256,tf32=False)\n    lock=dict(scope=\'Retrospective exploratory evaluation on previously opened sets; not independent confirmation\',methods=METHODS,seeds=SEEDS,models=[{k:v for k,v in m.items() if k not in [\'payload\',\'reference\']} for m in models],environment=env,input_anchors=anchors,contrasts=CONTRASTS,metrics=METRICS,bootstrap=\'Reuse 10000 fixed paired image draws; seeds fixed; no multiplicity adjustment\',student_TS=False,training=False,refit=False,replay_tolerance=1e-4)\n    out.mkdir(parents=True,exist_ok=True);lp=out/\'evaluation_LOCKED.json\'\n    if lp.exists():require(json.loads(lp.read_text())==lock,\'输出目录已有不同协议；请使用新的输出目录\')\n    else:save(lp,lock)\n    print(\'输入核验完成：12 个模型，1000 + 200 条；协议已锁定。\',flush=True)\n    return dict(ab=ab,ev=ev,out=out,models=models,data=datasets,dev=dev,device=device,lock=lock)\n\ndef predict(model,x,device):\n    import torch\n    with torch.inference_mode():return np.concatenate([model(torch.from_numpy(np.ascontiguousarray(x[i:i+256])).to(device)).cpu().numpy() for i in range(0,len(x),256)])\ndef load_head(m,device):\n    import torch\n    net=torch.nn.Sequential(torch.nn.Linear(2048,512),torch.nn.ReLU(),torch.nn.Dropout(.1),torch.nn.Linear(512,3))\n    net.load_state_dict(torch.load(io.BytesIO(m[\'payload\']),map_location=\'cpu\',weights_only=True),strict=True)\n    return net.to(device).eval()\ndef replay(ctx):\n    rows=[]\n    for m in ctx[\'models\']:\n        z=predict(load_head(m,ctx[\'device\']),ctx[\'dev\'][\'x\'],ctx[\'device\']);ref=m[\'reference\']\n        diff=float(np.max(abs(z-ref)));mismatch=int((z.argmax(1)!=ref.argmax(1)).sum())\n        require(diff<=1e-4 and mismatch==0,f"{m[\'model_id\']} 开发重放失败：{diff}, {mismatch}；不要放宽阈值。")\n        rows.append(dict(model_id=m[\'model_id\'],max_abs_difference=diff,class_mismatches=mismatch));print(\'Replay PASS\',m[\'model_id\'],flush=True)\n    table(ctx[\'out\']/\'development_replay.csv\',rows);ctx[\'replay_pass\']=True\n\ndef evaluate(ctx):\n    require(ctx.get(\'replay_pass\',False),\'必须先通过12个模型的开发重放\')\n    ctx[\'logits\']={};ctx[\'points\']={};records=[];summaries=[];cmrows=[]\n    for split,d in ctx[\'data\'].items():\n        predictions=[]\n        for m in ctx[\'models\']:\n            z=predict(load_head(m,ctx[\'device\']),d[\'x\'],ctx[\'device\']);vals,p,pred,_,_=metrics(z,d[\'y\'])\n            key=(split,m[\'method\'],m[\'seed\']);ctx[\'logits\'][key]=z;ctx[\'points\'][key]=vals\n            records.append(dict(split=split,method=m[\'method\'],seed=m[\'seed\'],variant=\'raw\',**vals))\n            folder=ctx[\'out\']/\'logits\'/split;folder.mkdir(parents=True,exist_ok=True)\n            np.savez_compressed(folder/(m[\'model_id\']+\'.npz\'),logits=z,ids=d[\'ids\'],image_ids=d[\'image_ids\'],y=d[\'y\'])\n            for i in range(len(z)):predictions.append(dict(model_id=m[\'model_id\'],sample_id=str(d[\'ids\'][i]),image_id=str(d[\'image_ids\'][i]),gold=int(d[\'y\'][i]),prediction=int(pred[i]),**{f\'logit_{c}\':float(z[i,j]) for j,c in enumerate(\'ENC\')},**{f\'p_{c}\':float(p[i,j]) for j,c in enumerate(\'ENC\')}))\n            cm=np.bincount(3*d[\'y\']+pred,minlength=9).reshape(3,3)\n            for i in range(3):\n                for j in range(3):cmrows.append(dict(split=split,method=m[\'method\'],seed=m[\'seed\'],gold=\'ENC\'[i],predicted=\'ENC\'[j],count=int(cm[i,j]),row_fraction=float(cm[i,j]/max(cm[i].sum(),1))))\n        table(ctx[\'out\']/split/\'predictions.csv\',predictions)\n        for method in METHODS:\n            rr=[r for r in records if r[\'split\']==split and r[\'method\']==method]\n            summaries.append(dict(split=split,method=method,n_seeds=3,**{k+s:float(fn([r[k] for r in rr])) for k in METRICS for s,fn in [(\'_mean\',np.mean),(\'_seed_sd\',lambda x:np.std(x,ddof=1))]}))\n        print(\'Evaluation complete:\',split,flush=True)\n    table(ctx[\'out\']/\'per_seed.csv\',records);table(ctx[\'out\']/\'summary.csv\',summaries);table(ctx[\'out\']/\'confusion_matrices.csv\',cmrows)\n    ctx[\'summary\']=summaries\n\ndef intervals(ctx):\n    rows=[];deltas=[]\n    for split,d in ctx[\'data\'].items():\n        order=d[\'order\'];y=d[\'y\'][order];scores={}\n        for m in ctx[\'models\']:\n            scores[m[\'method\'],m[\'seed\']]=bootstrap_scores(ctx[\'logits\'][split,m[\'method\'],m[\'seed\']][order],y,d[\'draws\'])\n            print(\'Bootstrap:\',split,m[\'model_id\'],flush=True)\n        for name,w in CONTRASTS.items():\n            for seed in SEEDS:\n                deltas.append(dict(split=split,comparison=name,seed=seed,**{k:sum(a*ctx[\'points\'][split,method,seed][k] for method,a in zip(METHODS,w)) for k in METRICS}))\n            for k in METRICS:\n                samples=np.mean([sum(a*scores[method,seed][k] for method,a in zip(METHODS,w)) for seed in SEEDS],axis=0)\n                values=[r[k] for r in deltas if r[\'split\']==split and r[\'comparison\']==name];lo,hi=np.quantile(samples,[.025,.975])\n                rows.append(dict(split=split,comparison=name,metric=k,delta=float(np.mean(values)),paired_seed_sd=float(np.std(values,ddof=1)),ci_low=float(lo),ci_high=float(hi),crosses_zero=bool(lo<=0<=hi),role=\'exploratory_unadjusted\',bootstrap_repeats=10000))\n    table(ctx[\'out\']/\'paired_intervals.csv\',rows);table(ctx[\'out\']/\'paired_seed_deltas.csv\',deltas);ctx[\'intervals\']=rows\n\ndef export(ctx):\n    out=ctx[\'out\'];rows=ctx[\'summary\'];ci=ctx[\'intervals\']\n    lines=[\'# 2×2统一评价结果\',\'\', \'四条件各三种子；固定已有checkpoint；未训练、未拟合温度、未做Student TS。\', \'\', \'历史1000与补充200均已有使用史。本次全部区间为探索性、未作多重比较校正；固定训练种子的图片bootstrap不消除选择偏差。\',\'\', \'| 集合 | 方法 | F1 % ± SD | NLL | ECE % | Brier |\',\'|---|---|---:|---:|---:|---:|\']\n    for r in rows:lines.append(f"| {r[\'split\']} | {r[\'method\']} | {100*r[\'macro_f1_mean\']:.3f} ± {100*r[\'macro_f1_seed_sd\']:.3f} | {r[\'nll_mean\']:.6f} | {100*r[\'ece10_mean\']:.3f} | {r[\'brier_mean\']:.6f} |")\n    lines+=[\'\',\'## 成分效应与交互\',\'\', \'| 集合 | 对比 | 指标 | 差值 [95%区间] |\',\'|---|---|---|---:|\']\n    for r in ci:\n        if r[\'metric\'] not in [\'macro_f1\',\'nll\',\'ece10\',\'brier\']:continue\n        s=100 if r[\'metric\'] in [\'macro_f1\',\'ece10\'] else 1\n        lines.append(f"| {r[\'split\']} | {r[\'comparison\']} | {r[\'metric\']} | {s*r[\'delta\']:+.5f} [{s*r[\'ci_low\']:+.5f}, {s*r[\'ci_high\']:+.5f}] |")\n    lines+=[\'\',\'F1/ECE差值展示为百分点。正差值利于F1，负差值利于NLL/ECE/Brier。交互=combined−bias−temperature+raw。区间跨0不证明等效；交互不跨0也不证明内部表征机制。不要将新模型结果替换旧主表而保留旧区间。\']\n    (out/\'RESULTS_结果解读.md\').write_text(\'\\n\'.join(lines),encoding=\'utf-8\')\n    # Self-contained tabular fragment, no additional package beyond booktabs.\n    tex=[\'\\\\begin{table*}[t]\',\'\\\\centering\\\\small\',\'\\\\begin{tabular}{llrrrr}\\\\toprule\',r\'Set & Target & F1 (\\%) & NLL & ECE (\\%) & Brier \\\\ \\midrule\']\n    for r in rows:\n        label=r[\'method\'].replace(\'_\',r\'\\_\');split=\'Historical\' if r[\'split\']==\'historical_test\' else \'Supplementary\'\n        tex.append(f"{split} & {label} & {100*r[\'macro_f1_mean\']:.3f} & {r[\'nll_mean\']:.6f} & {100*r[\'ece10_mean\']:.3f} & {r[\'brier_mean\']:.6f}"+r\' \\\\\')\n    tex += [r\'\\bottomrule\\end{tabular}\',r\'\\caption{Fixed 2$\\times$2 ablation on previously examined evaluation subsets. Three-seed means; no student TS. Full seed SDs and exploratory paired intervals accompany this table.}\',r\'\\end{table*}\']\n    (out/\'ablation_evaluation_table.tex\').write_text(\'\\n\'.join(tex),encoding=\'utf-8\')\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,axes=plt.subplots(1,2,figsize=(10,4.3));names=list(CONTRASTS)\n    for ax,metric in zip(axes,[\'macro_f1\',\'nll\']):\n        for j,(split,color) in enumerate([(\'historical_test\',\'#0072B2\'),(\'new_dev_holdout\',\'#D55E00\')]):\n            rr=[next(r for r in ci if r[\'split\']==split and r[\'metric\']==metric and r[\'comparison\']==n) for n in names];scale=100 if metric==\'macro_f1\' else 1\n            for i,r in enumerate(rr):\n                yy=i+(j-.5)*.18;ax.plot([r[\'ci_low\']*scale,r[\'ci_high\']*scale],[yy,yy],color=color);ax.plot(r[\'delta\']*scale,yy,\'o\' if j==0 else \'s\',color=color,label=split if i==0 else None)\n        ax.set_yticks(range(len(names)),[\'Bias at T=1\',\'Bias at T=T*\',\'T at b=0\',\'T at b=b*\',\'Combined - raw\',\'Interaction\']);ax.invert_yaxis();ax.axvline(0,ls=\'--\',color=\'gray\');ax.set_xlabel(\'Delta F1 (pp)\' if metric==\'macro_f1\' else \'Delta NLL\');ax.grid(axis=\'x\',alpha=.2)\n    axes[0].legend(fontsize=8);fig.suptitle(\'Exploratory 95% paired image-bootstrap intervals; fixed seeds\');fig.tight_layout()\n    for ext in [\'pdf\',\'png\']:fig.savefig(out/(\'factorial_intervals.\'+ext),dpi=200,bbox_inches=\'tight\')\n    plt.close(fig)\n    audit=dict(status=\'COMPLETE\',models=12,splits=2,metric_rows=24,prediction_rows=14400,interval_rows=len(ci),bootstrap_direct_checks=True,development_replay=True,created_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),scope=ctx[\'lock\'][\'scope\'])\n    save(out/\'completion.json\',audit)\n    hashes={p.relative_to(out).as_posix():digest_file(p) for p in sorted(out.rglob(\'*\')) if p.is_file() and p.name!=\'PACKAGE_SHA256.json\'}\n    save(out/\'PACKAGE_SHA256.json\',hashes)\n    archive=out.with_name(out.name+\'_results.zip\')\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(out.rglob(\'*\')):\n            if p.is_file():z.write(p,p.relative_to(out).as_posix())\n    print(\'完成。结果包：\',archive,flush=True);return archive\n'
INPUT_ANCHORS = {'ablation_protocol_sha256': '6079c58c8aeea255e401a6a9f5555f90523b038fe4f20e04d3d7c8319b050264', 'evaluation_protocol_sha256': 'f4e5acb3ca81cf8a2c27bd347b337c29cfaec9e6717c0793574ccd0a659ffe4e', 'evaluation_manifest_sha256': 'ac6915947711ffce7d18938e219df2ef4bb6f08370adc10ca0e1abf9374ed8d5', 'models': {'KD_raw_20261010': {'checkpoint_sha256': '4f2b53ea1887fd5d8630604fde0b07da46d2539f13f16adf5f9de6461053ec7b', 'best_epoch': 28, 'predictions_sha256': '64e754b9935ac7d61696018585c27c82dd2ac504cc318f54d2a708e3cb8fc4c2'}, 'KD_raw_20261011': {'checkpoint_sha256': '3d905cc39a34bab5a4a3021d82899be3d2c226e80a44a2060cad035f7d03e9fc', 'best_epoch': 21, 'predictions_sha256': 'fcef04ab8dbfd7cab802fefab493346894ac395e3566e58dc303d8272c3c280c'}, 'KD_raw_20261012': {'checkpoint_sha256': 'c1a157a362715deb96d1654cfead88c8d6e2c4506be34401210dfc19bfd016d5', 'best_epoch': 28, 'predictions_sha256': 'dcf1c4b7be143ac7556ff969e2cce454fa1ebe9196461a9ffcca861c3bf645ee'}, 'KD_bias_20261010': {'checkpoint_sha256': '995bdc60f942efdd1c15a76f42f80678f75cb265f0da38252232626c224a0391', 'best_epoch': 23, 'predictions_sha256': 'e2e4420721a1ed70912d0d390fb784a2af0e05bc1bc646db03856d857e93cd02'}, 'KD_bias_20261011': {'checkpoint_sha256': '17e7f127e231c513687dfdd60a512755750cc8929ac89dfc9c02de5304a6d1c8', 'best_epoch': 22, 'predictions_sha256': 'e6467a682ef31c05ff9568b6db3e4a54ad2fed5b081ac7ac565def06f23965be'}, 'KD_bias_20261012': {'checkpoint_sha256': '63cb1707523f4f4b194f8d9db81960da94a251dd7bbda6bc5bca93ac40c1283c', 'best_epoch': 21, 'predictions_sha256': '3628e8db7f3901a84e63c67da5fccc693a5c98b6eca1f9b2f5fc80f485a7bc65'}, 'KD_temperature_20261010': {'checkpoint_sha256': '54268dd7d79381e3ca302fcb516d7ee616deafdd3b178cf62d5c8e5c2f53c5b3', 'best_epoch': 23, 'predictions_sha256': '59143d5f02fbc963fa3ad31c012d533c996047c6712985127ee3bfb189d38f0d'}, 'KD_temperature_20261011': {'checkpoint_sha256': 'b3125ffe03be2918066b6a2611c8c4a765fd1d744c66e0216bc4bb5b49dadec0', 'best_epoch': 26, 'predictions_sha256': '26ec84b93c69de44b0ec9c5544d3546a12881e6a00031522ca39be256943421c'}, 'KD_temperature_20261012': {'checkpoint_sha256': '3013c5c9ba6f5f3dba89ee0019b33d5c570bfc4846c069f492ecb4cc2bbf22cd', 'best_epoch': 20, 'predictions_sha256': 'd730041dfb11e9a1dc36bd1a986aeb1155e3e515ccbb9dc043d6ef086367057d'}, 'KD_calibrated_20261010': {'checkpoint_sha256': 'c100fee94cf2d82c8e1656d4be468c1655c79a81817b678436c28e148c6a4a4f', 'best_epoch': 17, 'predictions_sha256': '01fa0f970700e36f7c2e4ed8a6ed6e9795b9d653a9b046cb7957f7af7e5da37f'}, 'KD_calibrated_20261011': {'checkpoint_sha256': 'c2a4ee53871bf0e222c045d8e0e9769573007ddc72a1a1ee698e9cd0f3010381', 'best_epoch': 21, 'predictions_sha256': '68fc38f23555c6bd86ec8a8ac3f5927f5861f1544634cd9cedc0a7678c30df21'}, 'KD_calibrated_20261012': {'checkpoint_sha256': '35b2d9d24c1b2926ce621fbfa428028cf7088e55dc39d7e2a2ec84338c5e5cec', 'best_epoch': 28, 'predictions_sha256': '3aafaff6a8e15e7d527203aed19deeb6c56a603bb3aed1f842d09eab05b2b79a'}}, 'evaluator_sha256': '05d52313ee54b2029a8855840f3a8636e35f9ce20bac60140e5f17ad97635270'}
import hashlib
assert hashlib.sha256(MODULE_SOURCE.encode('utf-8')).hexdigest() == INPUT_ANCHORS['evaluator_sha256']
exec(compile(MODULE_SOURCE, '<notebook_evaluator>', 'exec'))

## 5. 核验输入并锁定本轮评价
检查12个checkpoint、原训练协议、选定epoch、特征定义、ID/标签、两套评价集及bootstrap索引。固定所有比较后才运行模型。遇到错误请保留错误信息，不跳过核验。

In [ ]:
ctx = prepare(ABLATION_INPUT, EVALUATION_INPUT, OUTPUT_DIR, INPUT_ANCHORS, device=DEVICE)
(ctx['out']/'evaluate_2x2.py').write_text(MODULE_SOURCE, encoding='utf-8')
save(ctx['out']/'input_anchors.json', INPUT_ANCHORS)

## 6. 先重放开发集：12个模型全部通过后继续
将当前模型输出与消融包内已有的400条开发预测逐项比较：最大logit绝对误差≤1e−4，预测类别完全一致。此步骤核验模型加载与数值兼容性，不重新选模型。

In [ ]:
replay(ctx)

## 7. 运行历史1000条与补充200条评价
输出Macro-F1、Accuracy、各类F1、NLL、10-bin ECE、三分类Brier。CSV中F1/ECE为0–1比例；展示百分数时乘100。三种子分别计算后取均值和样本SD，不合并成集成模型。

In [ ]:
evaluate(ctx)
for r in ctx['summary']:
    print(f"{r['split']:18s} {r['method']:16s}  F1={100*r['macro_f1_mean']:.3f} ± {100*r['macro_f1_seed_sd']:.3f}%  NLL={r['nll_mean']:.6f}  ECE={100*r['ece10_mean']:.3f}%")

## 8. 配对区间：成分效应与交互

同一套10,000次图片抽样用于所有四条件和三个种子。每次先逐种子重算指标，再求配对差值和种子平均，最后取2.5/97.5百分位。

| 对比 | 定义 |
|---|---|
| 无温度时的偏置效应 | bias − raw |
| 有温度时的偏置效应 | combined − temperature |
| 无偏置时的温度效应 | temperature − raw |
| 有偏置时的温度效应 | combined − bias |
| 总体变化 | combined − raw |
| 交互（差分之差） | combined − bias − temperature + raw |

交互区间直接使用每次抽样的四条件联合差值计算，**不能用几个单独区间端点相减**。共两套数据×六种对比×八项指标=96行区间。此步会持续输出进度。

In [ ]:
intervals(ctx)

## 9. 生成图、LaTeX表、中文结果报告与ZIP
最终读四个重点：温度和偏置各自的条件效应；交互区间是否跨0；两套集合是否方向一致；开发观察是否延续。请完整保留有利、不利和跨0结果。

In [ ]:
RESULT_ZIP = export(ctx)
from IPython.display import display, Markdown, Image
display(Markdown((ctx['out']/'RESULTS_结果解读.md').read_text(encoding='utf-8')))
display(Image(filename=str(ctx['out']/'factorial_intervals.png')))
print('结果 ZIP：', RESULT_ZIP)

## 10. 下载结果（可选）
结果已保存到 Drive；也可运行本格直接下载。将生成的 results.zip 发回来即可据此修改论文。

In [ ]:
try:
    from google.colab import files
    files.download(str(RESULT_ZIP))
except ImportError:
    print('本地结果：', RESULT_ZIP)

## 常见问题

- **找不到输入**：改第2格路径。Google Drive里的文件名可能与下载到电脑时的文件名不同。
- **提示不是已核对版本/哈希不符**：确认使用2026-10-28的这份2×2结果，以及2026-10-26的统一评价协议输入包，不要改哈希跳过。
- **开发重放失败**：先检查运行环境；优先使用原PyTorch 2.11.0环境或CPU重试，并使用新输出目录。不要看完测试成绩再调阈值。
- **会重跑CLIP或Jev吗？** 不会，使用已冻结的2048维特征，只运行12个分类头。
- **会训练或拟合Student TS吗？** 不会。四条件评价统一使用原始Student logits（T=1）。
- **结果能直接覆盖旧主表吗？** 不能自动覆盖。新模型的checkpoint身份独立记录；论文须同时说明新旧模型关系，不将新点估计配上旧模型区间。
- **中断后怎么继续？** 同一会话中可从失败单元格重跑。运行时重置后从头运行，会产生新输出目录；保留先前结果以便追溯。